In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score,KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('time')
plt.ylabel('delivery')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(['Order_ID'], axis=1)
df_clean = df.copy()

In [ ]:
# Task 2: Write your code here:
missing_percentage = (df_clean.isnull().sum() / len(df_clean)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

missing_data.head(10)
df_clean = df_clean.dropna(missing_data.columns)
print(f"After dropping missing: {df_clean.shape}")

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(cols))
le = LabelEncoder()

# Encode the target column
label_encoders = {}
for col in cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])

df_clean.head()

In [ ]:
# Task 5: Write your code here:

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
Ratio = df_clean['Delivery_Time'].value_counts(normalize=True)
Ratio
#as we can see from the graph 📊 above the date is inbalisd

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("'Delivery_Time'", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)



In [ ]:
# Task 2,3,4,5: Write your code here:
# TODO: Split data with stratification (test_size=0.2, random_state=42)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

msn_s = []

for train_idx, val_idx in kfold.split(X):
    X_fold_train = X[train_idx]
    X_fold_val   = X[val_idx]

    y_fold_train = y.iloc[train_idx]
    y_fold_val   = y.iloc[val_idx]



    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    f1 = mean_absolute_error(y_fold_val, y_fold_pred, pos_label=1)
    msn_s.append(f1)
mae_scores = np.array(msn_s)


print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:

# Task 1: Write your code here:

coeffs = model.coef_


fig, axes = plt.subplots(1, 1, figsize=(15, 6))
axes = axes.flatten()
features = X.columns
i =0
for coef in coeffs.items:
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"random forst Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('time')
plt.ylabel('delivery')
plt.show()

In [ ]:
# Task Bonus: Write your code here: